# 00 – Study Area Exploration via PFAF_ID

**Purpose:** Defining a study area by selecting a Pfafstetter Level-6 basin from the global SWORD dataset.
This notebook is the single entry point for all subsequent analysis notebooks.
**Study area and PFAF_IDs are defined centrally in _00_config.py**. So manually steps are required, the notebook is fully automated.

**OUTPUT:** 
Overview of certain river system dimension.

**Concept:**
SWORD encodes the Pfafstetter Level-6 basin code directly in the first 6 digits of each `reach_id`:
```
reach_id format: CBBBBBRRRRT
  C      = Continent digit (first digit of Pfafstetter code)
  BBBBB  = Remaining Pfafstetter digits up to Level 6
  --> CBBBBB (first 6 digits) = Pfafstetter Level-6 basin code = PFAF_ID
  RRR    = Reach number within basin
  T      = Reach type (1=river, 3=lake, 4=dam, 5=unreliable, 6=ghost)
```
This means no spatial join is needed to assign a basin to a reach.
The PFAF_ID is extracted directly from `reach_id` as a new column.

**Workflow:**
```
0.0 Imports
0.1 Configuration  <-- only cell that needs editing between runs
0.2 Load global SWORD
0.3 Extract PFAF_ID column
0.4 Filter to study area
0.5 Validate & inspect
0.6 Save clipped SWORD (only for testing single rivers)
0.7 Interactive map
```

**Output:** `sword_{STUDY_AREA}_clip.gpkg` – clipped SWORD reaches for the chosen study area,
with `PFAF_ID` column added. This file is the input for Notebook 02 (Join).

---
## 0.0 | Imports

In [2]:
import geopandas as gpd
import pandas as pd
import os
import sys
import webbrowser

sys.path.append(os.path.join("..", "src"))

# Import from src files --> no manual configuration needed here
from _0_config_paths import (
    DATA_RAW, DATA_PROCESSED, DATA_OUTPUT,
    IN_SWORD, IN_BASIN_5,
    CONTINENT, REACH_TYPES_TO_KEEP
)
from _00_config import STUDY_AREA, PFAF_IDS, PFAF_LEVEL_DIGITS
from _00_setup import setup_study_area

SAGA found: C:\Program Files\QGIS 3.42.3\apps\saga7\saga_cmd.exe


---
## 0.1 | Configuration

To find the right PFAF_IDs for a river:
- Look up the first 6 digits of any known `reach_id` for that river (e.g. from Notebook 01)
- Or browse HydroSHEDS / HydroBASINS Level 6 to identify the relevant basin codes
- A river may span multiple Level-6 basins (e.g. a large river with major tributaries)
  --> list all relevant codes in `PFAF_IDS`



In [3]:
# ============================================================
# Configuration is centrally defined in src/_00_config.py
# Edit STUDY_AREA and PFAF_IDS there to switch rivers.
# ============================================================

# Output paths (notebook-specific)
OUT_SWORD_CLIP = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_clip.gpkg")
OUT_MAP        = os.path.join(DATA_OUTPUT,    f"00_{STUDY_AREA}_study_area_map.html")

# Summary
print(f"Study area  : {STUDY_AREA}")
print(f"PFAF_IDs    : {PFAF_IDS}")
print(f"Continent   : {CONTINENT}  (auto-detected from PFAF_ID)")
print(f"Reach types : {REACH_TYPES_TO_KEEP}")
print(f"IN_SWORD    : {IN_SWORD}")
print(f"Output      : {OUT_SWORD_CLIP}")
print()
print(f"Input exists: {os.path.exists(IN_SWORD)}")

Study area  : naryn
PFAF_IDs    : [46219]
Continent   : as  (auto-detected from PFAF_ID)
Reach types : [1, 3]
IN_SWORD    : D:\0_InnoLab\0_data\SWOT_river_database_SWORD\as_sword_reaches_v17b.gpkg
Output      : C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\processed\sword_naryn_clip.gpkg

Input exists: True


---
## 0.2 | Load SWORD

In [4]:
# Load HydroBASINS AOI polygon and filter SWORD reaches to study area
aoi, sword = setup_study_area(
    basin_path        = IN_BASIN_5,
    sword_path        = IN_SWORD,
    pfaf_ids          = PFAF_IDS,
    pfaf_level_digits = PFAF_LEVEL_DIGITS
)

# Derive bbox for all subsequent download functions
bbox = tuple(aoi.total_bounds)
print(f"\nStudy area ready: {len(sword)} reaches | bbox: {bbox}")

Loading HydroBASINS Level 5 from: D:\0_InnoLab\0_data\BasinATLAS_HydroSHEDS\BasinATLAS_v10_lev05.gpkg
Basin polygons found: 1 for PFAF_IDs: [46219]
AOI bbox: [71.3625     40.4375     78.35416667 42.46666667]

Loading SWORD from: D:\0_InnoLab\0_data\SWOT_river_database_SWORD\as_sword_reaches_v17b.gpkg
SWORD reaches in bbox: 214
SWORD reaches after PFAF_ID filter: 140

Study area ready: 140 reaches | bbox: (np.float64(71.36249999957539), np.float64(40.43750000022487), np.float64(78.35416666695818), np.float64(42.46666666673332))


---
## 0.3 | Define AOI based of PF

The Pfafstetter Level-6 code is encoded in the first 6 digits of `reach_id`.
It will be extracted as a dedicated integer column `PFAF_ID` so it can be used
for filtering, grouping, and downstream joins.

In [5]:
# Add PFAF_ID column (extracted from reach_id, first 5 digits)
sword["PFAF_ID"] = (
    sword["reach_id"]
    .astype(str)
    .str[:PFAF_LEVEL_DIGITS]
    .astype(int)
)

# Add reach_type column (last digit of reach_id)
# Type: 1=river, 3=lake on river, 4=dam/waterfall, 5=unreliable, 6=ghost
sword["reach_type"] = (
    sword["reach_id"]
    .astype(str)
    .str[-1]
    .astype(int)
)

print(f"PFAF_ID column added. Unique basins: {sword['PFAF_ID'].nunique()}")
print(f"\nReach type distribution:")
print(sword["reach_type"].value_counts().sort_index())

PFAF_ID column added. Unique basins: 1

Reach type distribution:
reach_type
1    106
3     16
4      5
6     13
Name: count, dtype: int64


---
## 0.4 | Filter to study area

Filtering the global dataset to the PFAF_IDs defined in Section 0.1.

In [6]:
# Filter by reach type (defined in _00_config.py: REACH_TYPES_TO_KEEP)
sword_clip = sword[sword["reach_type"].isin(REACH_TYPES_TO_KEEP)].copy()
sword_clip = sword_clip.reset_index(drop=True)

print(f"Reaches in study area  : {len(sword_clip):,}")
print(f"Reaches excluded (type): {len(sword) - len(sword_clip):,}")
print(f"PFAF_IDs found         : {sorted(sword_clip['PFAF_ID'].unique())}")
print(f"Reach types kept       : {sorted(sword_clip['reach_type'].unique())}")

# Sanity check: warn if any requested PFAF_ID returned zero reaches
for pfaf in PFAF_IDS:
    n = (sword_clip["PFAF_ID"] == pfaf).sum()
    if n == 0:
        print(f"  WARNING: PFAF_ID {pfaf} matched 0 reaches")
    else:
        print(f"  PFAF_ID {pfaf}: {n} reaches")

Reaches in study area  : 122
Reaches excluded (type): 18
PFAF_IDs found         : [np.int64(46219)]
Reach types kept       : [np.int64(1), np.int64(3)]
  PFAF_ID 46219: 122 reaches
